# `main.ipynb` — Pipeline ETL GeoStat (paquete `geostat_etl`)

**Autor:** Francisco Máximo Ortega Calvo
**Área:** Data & BI

Punto de entrada del proyecto. El pipeline está organizado como un conjunto de **notebooks-módulo** dentro de `geostat_etl/`, uno por responsabilidad, cada uno con su propio markdown documentado y sus celdas de prueba. Como un `.ipynb` no se puede `import`ar como un `.py`, este notebook los encadena con la magia `%run` (en orden de dependencias) y luego ejecuta el pipeline completo.

Al ejecutar la sección 1 se verá en vivo, módulo a módulo, **todas las pruebas y el logging** de cada pieza: configuración, funciones de limpieza, la llamada real a la API de demografía (con sus warnings y la activación del fallback si aplica), la conexión a PostgreSQL, la extracción sobre la SQLite real, la consolidación con datos de ejemplo y la definición del orquestador.

| Notebook-módulo (`geostat_etl/`) | Responsabilidad |
|---|---|
| `config.ipynb` | Credenciales, rutas y reglas de negocio parametrizadas |
| `logging_config.ipynb` | Configuración centralizada del logging |
| `transform.ipynb` | Limpieza, normalización y validación |
| `demografia.ipynb` | Extracción resiliente de la API REST (+ fallback interno) |
| `db.ipynb` | Conexión a PostgreSQL y auditoría |
| `extract.ipynb` | Extracción y limpieza de la fuente Legacy SQLite |
| `consolidate.ipynb` | Consolidación estadística y cálculo de indicadores |
| `load.ipynb` | Carga idempotente en PostgreSQL (UPSERT) + cuarentena |
| `pipeline.ipynb` | Orquestador `ejecutar_pipeline()` |

| Sección de este notebook | Contenido |
|---|---|
| 1 | Cargar los módulos (`%run`) - dispara todas sus pruebas internas |
| 2 | Ejecución del pipeline completo |
| 3 | Resultados |
| 4 | Verificación final en PostgreSQL |

---
## 1. Cargar los módulos

Este notebook vive en la **raíz del proyecto**, y `geostat_etl/` es una subcarpeta — de ahí las rutas `geostat_etl/xxx.ipynb` en cada `%run`. El orden importa: cada módulo usa nombres definidos por el anterior (p. ej. `transform.ipynb` necesita las constantes de `config.ipynb`; `demografia.ipynb` necesita `logger` de `logging_config.ipynb` y `normalizar_nombre_pais` de `transform.ipynb`).

Al ejecutar cada celda se verá el resultado de la celda de "prueba rápida" de ese módulo - es la forma de comprobar, módulo a módulo, que todo funciona antes de lanzar el pipeline completo.

In [57]:
%run geostat_etl/config.ipynb

Destino PostgreSQL configurado -> host=localhost, puerto=5433, bd=db_geostat_dw, usuario=geostat_user
Origen SQLite -> ./datos_economicos_locales.db (tabla: economia_paises), chunk_size=2000
Dataset de respaldo cargado: 44 paises europeos


In [58]:
%run geostat_etl/logging_config.ipynb

2026-09-10 10:40:58 - INFO - Prueba de logging desde logging_config.ipynb


In [59]:
%run geostat_etl/transform.ipynb

'  Cyprus  '         -> 'CYPRUS'
'ESPAÑA'             -> 'SPAIN'
'francia'            -> 'FRANCE'
'  montenegro'       -> 'MONTENEGRO'
'BULGARIA'           -> 'BULGARIA'
2584.81
56594.0
861.03 EUR
1000.0 EUR
ValueError esperado: Divisa no soportada en la estructura de cambio: JPY
True False
False
True


In [60]:
%run geostat_etl/demografia.ipynb


Origen: FALLBACK
Paises disponibles: 44
Ejemplo: {'ALBANIA': 2771508, 'ANDORRA': 82904, 'AUSTRIA': 9113574}


In [61]:
%run geostat_etl/db.ipynb

Conexion OK -> PostgreSQL 15.19 (Debian 15.19-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [62]:
%run geostat_etl/extract.ipynb

Total leido: 20000
Validos: 18798
Cuarentena: 1202

Primeras filas validas:
nombre_pais  superficie_km2  pib_total_eur
 LUXEMBOURG        2584.810   8.127716e+10
    CROATIA       56594.000   7.211870e+10
    ICELAND      102998.722   2.429034e+10


In [63]:
%run geostat_etl/consolidate.ipynb

nombre_pais  superficie_km2  pib_total_eur  n_reportes
     FRANCE        643801.0   2.790000e+12           3
     MONACO             2.0   8.520000e+09           3

nombre_pais  superficie_km2  pib_total_eur  n_reportes  poblacion_total  densidad_poblacional  pib_per_capita_eur fuente_poblacion
     France        643801.0   2.790000e+12           3         66650804               103.527            41859.96         FALLBACK
     Monaco             2.0   8.520000e+09           3            38341             19170.500           222216.43         FALLBACK


In [64]:
%run geostat_etl/load.ipynb

In [65]:
%run geostat_etl/pipeline.ipynb

`logging_config.ipynb` ya dejó configurado el logging, pero su propia celda de prueba escribió una línea de prueba en `logs/ejecucion_etl.log`. Se vuelve a llamar a `configurar_logging()` aquí para dejar el log **limpio** antes de la ejecución real del pipeline (recuerda: `filemode='w'` trunca el fichero en cada llamada).

In [66]:
logger = configurar_logging()
print(f"Todos los modulos cargados. Logging reiniciado limpio -> {RUTA_LOG}")

Todos los modulos cargados. Logging reiniciado limpio -> ./logs/ejecucion_etl.log


---
## 2. Ejecución del pipeline completo

`ejecutar_pipeline()` (definida en `pipeline.ipynb`) encadena extracción demográfica, extracción + limpieza de la SQLite, consolidación estadística, cálculo de indicadores y carga idempotente en PostgreSQL, dejando auditoría completa en `tb_ejecuciones_etl`. Se puede observar en la salida los `INFO` de cada chunk procesado y, si aplica, los `WARNING` de la API.

In [67]:
df_indicadores, df_cuarentena = ejecutar_pipeline()

print(f"\nTotal paises cargados en tb_indicadores_europa: {len(df_indicadores)}")
print(f"Total registros desviados a tb_cuarentena_geodatos: {len(df_cuarentena)}")


Total paises cargados en tb_indicadores_europa: 44
Total registros desviados a tb_cuarentena_geodatos: 1202


---
## 3. Resultados

Top 10 países por PIB per cápita (para verificar visualmente que Mónaco, Liechtenstein y Luxemburgo aparecen en las posiciones superiores, tal como exige la especificación) y desglose de la cuarentena por motivo de rechazo.

In [68]:
print(df_indicadores.sort_values("pib_per_capita_eur", ascending=False)[
    ["nombre_pais", "poblacion_total", "superficie_km2", "pib_total_eur",
     "densidad_poblacional", "pib_per_capita_eur", "fuente_poblacion"]
].head(10).to_string(index=False))

  nombre_pais  poblacion_total  superficie_km2  pib_total_eur  densidad_poblacional  pib_per_capita_eur fuente_poblacion
       Monaco            38341           2.000   8.516855e+09            19170.5000           222134.40         FALLBACK
Liechtenstein            40128         160.579   7.434190e+09              249.8957           185261.91         FALLBACK
   Luxembourg           680454        2584.810   8.193110e+10              263.2511           120406.52         FALLBACK
      Ireland          5308039       70273.000   5.298043e+11               75.5345            99811.68         FALLBACK
  Switzerland          8967408       41284.441   8.476641e+11              217.2104            94527.21         FALLBACK
       Norway          5618354      385178.133   5.132544e+11               14.5864            91353.17         FALLBACK
      Denmark          6002507       42895.414   3.748757e+11              139.9335            62453.19         FALLBACK
      Iceland           398266  

In [69]:
print(df_cuarentena["motivo_rechazo"].value_counts().to_string())

motivo_rechazo
Entidad territorial no perteneciente a la region europea    993
Valor de PIB incoherente (PIB <= 0)                         209


---
## 4. Verificación final en PostgreSQL

Consulta directa a PostgreSQL (no al DataFrame en memoria) para confirmar el resultado esperado por la especificación: **exactamente 44 registros** en `tb_indicadores_europa`.

In [70]:
conn_verificacion = conectar_postgres()

print("=== tb_indicadores_europa (total de paises) ===")
print(pd.read_sql_query("SELECT COUNT(*) AS total_paises FROM tb_indicadores_europa;", conn_verificacion))

print("\n=== Top 10 por PIB per capita ===")
print(pd.read_sql_query(
    '''SELECT id, nombre_pais, poblacion_total, superficie_km2, pib_total_eur,
              densidad_poblacional, pib_per_capita_eur
       FROM tb_indicadores_europa
       ORDER BY pib_per_capita_eur DESC
       LIMIT 10;''',
    conn_verificacion,
))

print("\n=== tb_cuarentena_geodatos (motivos de rechazo) ===")
print(pd.read_sql_query(
    "SELECT motivo_rechazo, COUNT(*) AS total FROM tb_cuarentena_geodatos GROUP BY motivo_rechazo;",
    conn_verificacion,
))

print("\n=== tb_ejecuciones_etl (ultima ejecucion) ===")
print(pd.read_sql_query(
    "SELECT * FROM tb_ejecuciones_etl ORDER BY id_ejecucion DESC LIMIT 1;",
    conn_verificacion,
))

conn_verificacion.close()

=== tb_indicadores_europa (total de paises) ===
   total_paises
0            44

=== Top 10 por PIB per capita ===
   id    nombre_pais  poblacion_total  superficie_km2  pib_total_eur  \
0  27         Monaco            38341           2.000   8.516855e+09   
1  22  Liechtenstein            40128         160.579   7.434190e+09   
2  24     Luxembourg           680454        2584.810   8.193110e+10   
3  19        Ireland          5308039       70273.000   5.298043e+11   
4  41    Switzerland          8967408       41284.441   8.476641e+11   
5  31         Norway          5618354      385178.133   5.132544e+11   
6  11        Denmark          6002507       42895.414   3.748757e+11   
7  18        Iceland           398266      102998.722   2.412383e+10   
8  29    Netherlands         18346819       41543.000   9.909740e+11   
9  35     San Marino            33572          61.000   1.800207e+09   

   densidad_poblacional  pib_per_capita_eur  
0            19170.5000           222134.40  


C:\Users\EM2026008667\AppData\Local\Temp\ipykernel_14676\1108268234.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query("SELECT COUNT(*) AS total_paises FROM tb_indicadores_europa;", conn_verificacion))
C:\Users\EM2026008667\AppData\Local\Temp\ipykernel_14676\1108268234.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(
C:\Users\EM2026008667\AppData\Local\Temp\ipykernel_14676\1108268234.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(
C:\Users\EM